# Lab 20-04 · An LDO across Load, Capacitor and Supply Ripple

**Predict first**, then run. The handout is [README.md](README.md). The pass device is the chapter's illustrative T130 PMOS.

In [1]:
import math
import re
import subprocess
from dataclasses import replace
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import ldo

reg = ldo.design()
loads = {100e-6: "100 uA", 1e-3: "1 mA", 10e-3: "10 mA", 100e-3: "100 mA"}
f = np.logspace(0, 9, 1201)


def ngspice(netlist):
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {m.group(1): float(m.group(2)) for m in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}


print(f"W = {reg.w * 1e3:.3f} mm, C_g = {reg.cg * 1e12:.3f} pF")

W = 1.877 mm, C_g = 1.614 pF


## Part 1 · Sizing for dropout

*Predict* how much wider the pass device must be for 100 mV of dropout from 1.3 V instead of 150 mV from 1.35 V.

In [2]:
for vin_min, vdo in ((1.35, 0.15), (1.3, 0.10)):
    sz = ldo.size_pass_device(reg.dev, vin_min, vdo, 0.1)
    print(f"{vdo * 1e3:.0f} mV from {vin_min} V: W_long = {sz['w_long'] * 1e3:.3f} mm, factor = {sz['vs_factor']:.3f}, "
          f"W = {sz['w'] * 1e3:.3f} mm, C_g = {sz['cgg'] * 1e12:.3f} pF")

150 mV from 1.35 V: W_long = 1.609 mm, factor = 1.167, W = 1.877 mm, C_g = 1.614 pF
100 mV from 1.3 V: W_long = 2.470 mm, factor = 1.111, W = 2.745 mm, C_g = 2.361 pF


## Part 2 · Stability across load

*Predict* whether the phase margin is worst at light load or at full load.

In [3]:
fig, (a1, a2) = plt.subplots(2, 1, sharex=True, figsize=(7, 6))
for il, lab in loads.items():
    loop, *_ = reg.transfer(il)
    hv = np.array([loop(2j * math.pi * x) for x in f])
    m = ldo.margins(loop)
    print(f"{lab:>7}: L(0) = {m['l0']:7.1f}, f_c = {m['fc'] / 1e3:9.2f} kHz, PM = {m['pm']:.1f} deg")
    a1.semilogx(f, 20 * np.log10(abs(hv)), label=lab)
    a2.semilogx(f, np.degrees(np.unwrap(np.angle(hv))))
a1.axhline(0, color="k", lw=0.8); a1.set_ylabel("|L| (dB)"); a1.legend(); a2.set_ylabel("phase (deg)"); a2.set_xlabel("Hz")
plt.show()
print(ngspice("ldo_loop.cir"))

 100 uA: L(0) =   844.0, f_c =     22.50 kHz, PM = 89.9 deg
   1 mA: L(0) =   636.7, f_c =    168.93 kHz, PM = 89.1 deg
  10 mA: L(0) =   295.2, f_c =    779.60 kHz, PM = 85.8 deg
 100 mA: L(0) =    94.1, f_c =   2394.44 kHz, PM = 77.5 deg


{'l0a': 843.9672, 'l0b': 94.09114, 'fca': 22498.78, 'pha': -1.571846, 'fcb': 2394450.0, 'phb': -1.789065, 'fcc': 12999800.0, 'phc': -2.596914, 'pma': 89.93986, 'pmb': 77.49413, 'pmc': 31.20779}


## Part 3 · Supply rejection

*Predict* the DC PSRR at 100 mA with the gate tracking the supply (k = 1) and with a ground-referenced gate (k = 0).

In [4]:
plt.figure()
for il, k in ((100e-3, 1.0), (100e-3, 0.0), (100e-6, 1.0), (100e-6, 0.0)):
    _, _, _, pinv, _ = replace(reg, k_gate=k).transfer(il)
    vals = np.array([-20 * math.log10(abs(pinv(2j * math.pi * x))) for x in f])
    print(f"{loads[il]:>7}, k = {k:.0f}: {vals[0]:.1f} dB at 1 Hz, {np.interp(6, np.log10(f), vals):.1f} dB at 1 MHz")
    plt.semilogx(f, vals, label=f"{loads[il]}, k = {k:.0f}")
plt.xlabel("Hz"); plt.ylabel("PSRR (dB)"); plt.legend(); plt.show()
print(ngspice("ldo_psrr.cir"))

 100 mA, k = 1: 45.6 dB at 1 Hz, 45.9 dB at 1 MHz
 100 mA, k = 0: 32.0 dB at 1 Hz, 32.3 dB at 1 MHz
 100 uA, k = 1: 64.6 dB at 1 Hz, 97.5 dB at 1 MHz
 100 uA, k = 0: 33.7 dB at 1 Hz, 66.7 dB at 1 MHz
{'r1dc': 45.58344, 'r1m': 45.85714, 'r10m': 56.34528, 'r0dc': 32.02462, 'r0m': 32.29832, 'r010m': 42.78646, 'rlight': 64.60072}


## Part 4 · Load steps

*Predict* the dip for a 100 uA to 100 mA step with 1 uF, then with 0.1 uF. Each nonlinear run takes several seconds.

In [5]:
plt.figure()
for i0 in (100e-6, 1e-3, 10e-3):
    r = ldo.load_step_nonlinear(reg, i0, 0.1, t_end=3e-6, npts=30001)
    print(f"from {i0 * 1e3:6.3f} mA: minimum {r['peak'] * 1e3:.2f} mV at {r['t_peak'] * 1e6:.3f} us, end {r['final'] * 1e3:.2f} mV")
    plt.plot(r["t"] * 1e6, r["v"] * 1e3, label=f"from {i0 * 1e3:g} mA, 1 uF")
r = ldo.load_step_nonlinear(replace(reg, cl=0.1e-6), 100e-6, 0.1, t_end=1e-6, npts=100001)
print(f"0.1 uF: minimum {r['peak'] * 1e3:.1f} mV at {r['t_peak'] * 1e9:.1f} ns, end {r['final'] * 1e3:.2f} mV")
plt.plot(r["t"] * 1e6, r["v"] * 1e3, label="from 0.1 mA, 0.1 uF")
plt.xlim(0, 1); plt.xlabel("time (us)"); plt.ylabel("output deviation (mV)"); plt.legend(); plt.show()

from  0.100 mA: minimum -13.43 mV at 0.304 us, end -13.41 mV


from  1.000 mA: minimum -11.62 mV at 0.288 us, end -11.61 mV


from 10.000 mA: minimum -8.50 mV at 0.261 us, end -8.48 mV


0.1 uF: minimum -24.0 mV at 31.6 ns, end -13.41 mV


## Part 5 · A slower error amplifier

*Predict* the full-load phase margin and the 1 MHz PSRR when C_a grows from 0.2 pF to 2 pF.

In [6]:
slow = replace(reg, ca=2e-12)
for name, r_ in (("C_a = 0.2 pF", reg), ("C_a = 2 pF", slow)):
    loop, _, _, pinv, _ = r_.transfer(0.1)
    m = ldo.margins(loop)
    print(f"{name}: f_c = {m['fc'] / 1e6:.2f} MHz, PM = {m['pm']:.1f} deg, "
          f"PSRR(1 MHz) = {-20 * math.log10(abs(pinv(2j * math.pi * 1e6))):.1f} dB")

C_a = 0.2 pF: f_c = 2.39 MHz, PM = 77.5 deg, PSRR(1 MHz) = 45.9 dB
C_a = 2 pF: f_c = 1.30 MHz, PM = 34.6 deg, PSRR(1 MHz) = 37.9 dB
